# Data Exploration & Model Prototyping
Quick EDA and baseline model evaluation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = '../ml/data/raw/dataset.csv'
df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# Basic info
df.info()
df.describe()

In [ ]:
# Missing values
missing = df.isnull().sum()
print(missing[missing > 0])

In [ ]:
# Target distribution (update 'label' to your actual target column)
TARGET = 'label'
if TARGET in df.columns:
    df[TARGET].value_counts().plot(kind='bar', title='Target Distribution')
    plt.tight_layout()
    plt.show()

In [ ]:
# Baseline model
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

df_clean = df.dropna()
X = df_clean.drop(columns=[TARGET])
y = df_clean[TARGET]

# Keep only numeric columns for quick baseline
X = X.select_dtypes(include=[np.number])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model.fit(X_train, y_train)
print(classification_report(y_test, model.predict(X_test)))

In [ ]:
# Feature importance
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
importances.head(10).plot(kind='bar', title='Top 10 Feature Importances')
plt.tight_layout()
plt.show()

In [ ]:
# Evidently drift check (reference vs current — using same data as placeholder)
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset

report = Report(metrics=[DataDriftPreset()])
report.run(reference_data=X_train, current_data=X_test)
report.show(mode='inline')